# Final project: analyze dataset infectious_cases in PostgreSQL

## Setup environment

PostgreSQL runs within pgserver in `/tmp/final_project`. The notebook can be used in Google Colab and locally with Python 3.10.

In [1]:
# Setup the environment
%pip install -q pgserver psycopg2-binary sqlalchemy pandas


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import pgserver
from pathlib import Path
from sqlalchemy import create_engine, text
from urllib.request import urlopen

pg = pgserver.get_server('/tmp/final_project', cleanup_mode='stop')

engine = create_engine(
    pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1),
    future=True,
    connect_args={'options': '-c timezone=UTC -c search_path=pandemic'},
)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [3]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

def run(sql_request: str) -> None:
    """Run a multi-statement script on a raw psycopg2 cursor."""
    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cur:
            cur.execute(sql_request)
        raw_conn.commit()
    finally:
        raw_conn.close()

def show(sql_request: str) -> pd.DataFrame:
    """Run a SELECT and return result as a table."""
    with engine.begin() as conn:
        result = pd.read_sql_query(text(sql_request), conn, coerce_float=False)
    
    return result.astype(object).where(result.notna(), None)

## 1. Load data

**1.1 Create the pandemic schema and set it as default**

The same `search_path` is also set for every pooled connection in the engine options, so it does not depend on which connection a query happens to use.

In [4]:
run(r"""
    CREATE SCHEMA IF NOT EXISTS pandemic;
    SET search_path TO pandemic;
""")

show(r"""
    SELECT current_schema() AS default_schema,
           current_setting('search_path') AS search_path,
           current_setting('server_version') AS server_version;
""")

,default_schema,search_path,server_version
0,pandemic,pandemic,16.2


**1.2 Import csv to table infectious_cases**

In [5]:
FILE_ID = '1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl'
URL = f'https://drive.usercontent.google.com/download?id={FILE_ID}&export=download&confirm=t'
CSV_PATH = Path('data/infectious_cases.csv')

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

if not CSV_PATH.exists():
    with urlopen(URL, timeout=120) as response:
        payload = response.read()
    CSV_PATH.write_bytes(payload)

print(CSV_PATH.name, CSV_PATH.stat().st_size, 'bytes')

infectious_cases.csv 540895 bytes


In [6]:
run(r"""
    DROP TABLE IF EXISTS infectious_cases CASCADE;

    CREATE TABLE infectious_cases (
        entity               TEXT,
        code                 TEXT,
        year                 TEXT,
        number_yaws          TEXT,
        polio_cases          TEXT,
        cases_guinea_worm    TEXT,
        number_rabies        TEXT,
        number_malaria       TEXT,
        number_hiv           TEXT,
        number_tuberculosis  TEXT,
        number_smallpox      TEXT,
        number_cholera_cases TEXT
    );
""")

print("Infectious cases table created successfully!")


Infectious cases table created successfully!


In [7]:
raw_df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
raw_df.columns = raw_df.columns.str.lower()

with engine.begin() as conn:
    raw_df.to_sql('infectious_cases', conn, schema='pandemic', if_exists='append',
                  index=False, chunksize=2_000)

print('Rows in CSV:', len(raw_df))

Rows in CSV: 10521


**1.3 Review data**

In [8]:
show("""
    SELECT *
    FROM infectious_cases
    LIMIT 10;
""")

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,,6160,0,,,,,,
1,Afghanistan,AFG,1981,,5859,0,,,,,,
2,Afghanistan,AFG,1982,,9730,0,,,,,,
3,Afghanistan,AFG,1983,,13937,0,,,,,,
4,Afghanistan,AFG,1984,,3864,0,,,,,,
5,Afghanistan,AFG,1985,,13867,0,,,,,,
6,Afghanistan,AFG,1986,,12901,0,,,,,,
7,Afghanistan,AFG,1987,,4396,0,,,,,,
8,Afghanistan,AFG,1988,,2149,0,,,,,,
9,Afghanistan,AFG,1989,,385,0,,,,,,


In [9]:
show(r"""
    SELECT
        COUNT(*)                                           AS total_rows,
        COUNT(DISTINCT entity)                             AS total_entities,
        COUNT(DISTINCT NULLIF(code, ''))                   AS total_codes,
        MIN(year::INTEGER)                                 AS min_year,
        MAX(year::INTEGER)                                 AS max_year,
        COUNT(*) FILTER (WHERE code = '')                  AS empty_code,
        COUNT(*) FILTER (WHERE number_rabies = '')         AS empty_rabies,
        COUNT(*) FILTER (WHERE number_malaria = '')        AS empty_malaria,
        COUNT(*) FILTER (WHERE polio_cases = '')           AS empty_polio,
        COUNT(*) FILTER (WHERE number_yaws = '')           AS empty_yaws
    FROM infectious_cases;
""")

,total_rows,total_entities,total_codes,min_year,max_year,empty_code,empty_rabies,empty_malaria,empty_polio,empty_yaws
0,10521,245,211,1920,2023,1128,3741,3741,2911,10447


In [10]:
show(r"""
SELECT t.column_name,
       COUNT(*) FILTER (WHERE t.value = '') AS blank,
       COUNT(*) FILTER (WHERE t.value ~ '^[0-9]+(\.[0-9]+)?$') AS numeric_values,
       COUNT(*) FILTER (WHERE t.value <> '' AND t.value !~ '^[0-9]+(\.[0-9]+)?$') AS other_values
FROM infectious_cases
CROSS JOIN LATERAL (
    VALUES
        ('year', year),
        ('number_yaws', number_yaws),
        ('polio_cases', polio_cases),
        ('cases_guinea_worm', cases_guinea_worm),
        ('number_rabies', number_rabies),
        ('number_malaria', number_malaria),
        ('number_hiv', number_hiv),
        ('number_tuberculosis', number_tuberculosis),
        ('number_smallpox', number_smallpox),
        ('number_cholera_cases', number_cholera_cases)
) AS t (column_name, value)
GROUP BY t.column_name
ORDER BY t.column_name;
""")

,column_name,blank,numeric_values,other_values
0,cases_guinea_worm,2093,8428,0
1,number_cholera_cases,7554,2967,0
2,number_hiv,3741,6780,0
3,number_malaria,3741,6780,0
4,number_rabies,3741,6780,0
5,number_smallpox,10417,104,0
6,number_tuberculosis,3741,6780,0
7,number_yaws,10447,74,0
8,polio_cases,2911,7610,0
9,year,0,10521,0


Conclusion:
* Import: 10,521 rows were loaded into pandemic.infectious_cases - 245 entities for the years 1920–2023.
* Missing values are encoded as empty strings: Code is empty in 1,128 rows. Number_rabies, Number_malaria, Number_hiv and Number_tuberculosis are empty in the same 3,741 rows. Number_yaws is filled in only 74 of 10,521 rows.
* The year and indicator columns contain only empty strings and numbers (other_values = 0), so NULLIF(value, '')::NUMERIC runs without errors.

## 2. Normalize to 3NF

The key of the original table is `(Entity, Year)`. Functional dependencies:

* `Entity → Code`: every entity has exactly one code;
* `(Entity, Year) → Number_*`: the indicators of a specific year.

`Code` depends only on a part of the key (`Entity`). This partial dependency violates 2NF, and therefore 3NF as well. In practice the pair `(Entity, Code)` is repeated for every year of an entity.

Decomposition:

* `entities (entity_id PK, entity_name UNIQUE, entity_code UNIQUE)` - unique combinations of Entity + Code;
* `disease_cases (entity_id FK, year, indicators; PK (entity_id, year))` - indicators, where `entity_id` replaces the repeated Entity and Code.

Both tables are in 3NF: in `entities` every attribute depends only on `entity_id`; in `disease_cases` every indicator depends on the full key `(entity_id, year)` and there are no dependencies between non-key attributes.

The indicators are kept in a "wide" format (one column per disease) instead of an EAV layout `(entity_id, year, disease, value)`. Each disease is a separate attribute of the "entity-year" fact, so 3NF does not require unpivoting, and the wide format keeps proper types, `CHECK` constraints and simple queries.

Empty strings of the raw table become `NULL`, and the text values are cast to `NUMERIC` / `SMALLINT`.

In [11]:
run(r"""
    DROP TABLE IF EXISTS disease_cases CASCADE;
    DROP TABLE IF EXISTS entities CASCADE;
    
    CREATE TABLE entities (
        entity_id    INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        entity_name  TEXT NOT NULL UNIQUE,
        entity_code  TEXT UNIQUE
    );

    CREATE TABLE disease_cases (
        entity_id            INTEGER  NOT NULL REFERENCES entities (entity_id),
        year                 SMALLINT NOT NULL,
        number_yaws          NUMERIC CHECK (number_yaws          >= 0),
        polio_cases          NUMERIC CHECK (polio_cases          >= 0),
        cases_guinea_worm    NUMERIC CHECK (cases_guinea_worm    >= 0),
        number_rabies        NUMERIC CHECK (number_rabies        >= 0),
        number_malaria       NUMERIC CHECK (number_malaria       >= 0),
        number_hiv           NUMERIC CHECK (number_hiv           >= 0),
        number_tuberculosis  NUMERIC CHECK (number_tuberculosis  >= 0),
        number_smallpox      NUMERIC CHECK (number_smallpox      >= 0),
        number_cholera_cases NUMERIC CHECK (number_cholera_cases >= 0),
        PRIMARY KEY (entity_id, year)
    );
""")

In [12]:
run(r"""
    INSERT INTO entities (entity_name, entity_code)
    SELECT DISTINCT entity, NULLIF(code, '')
    FROM infectious_cases
    ORDER BY entity;

    INSERT INTO disease_cases (
        entity_id,
        year,
        number_yaws, 
        polio_cases,
        cases_guinea_worm,
        number_rabies,
        number_malaria,
        number_hiv,
        number_tuberculosis,
        number_smallpox,
        number_cholera_cases
    )
    SELECT
        e.entity_id,
        ic.year::SMALLINT,
        NULLIF(ic.number_yaws, '')::NUMERIC,
        NULLIF(ic.polio_cases, '')::NUMERIC,
        NULLIF(ic.cases_guinea_worm, '')::NUMERIC,
        NULLIF(ic.number_rabies, '')::NUMERIC,
        NULLIF(ic.number_malaria, '')::NUMERIC,
        NULLIF(ic.number_hiv, '')::NUMERIC,
        NULLIF(ic.number_tuberculosis, '')::NUMERIC,
        NULLIF(ic.number_smallpox, '')::NUMERIC,
        NULLIF(ic.number_cholera_cases, '')::NUMERIC
    FROM infectious_cases AS ic
    JOIN entities AS e
        ON e.entity_name = ic.entity;
""")

In [13]:
show(r"""
    SELECT 'infectious_cases (raw)' AS table_name, COUNT(*) AS n_rows 
    FROM infectious_cases
    UNION ALL
    SELECT 'entities', COUNT(*) 
    FROM entities
    UNION ALL
    SELECT 'disease_cases', COUNT(*) 
    FROM disease_cases;
""")

,table_name,n_rows
0,infectious_cases (raw),10521
1,entities,245
2,disease_cases,10521


In [14]:
show("""
    SELECT *
    FROM entities
    ORDER BY entity_id
    LIMIT 10;
""")

,entity_id,entity_name,entity_code
0,1,Afghanistan,AFG
1,2,Africa,None
2,3,African Region (WHO),None
3,4,Albania,ALB
4,5,Algeria,DZA
5,6,American Samoa,ASM
6,7,Americas,None
7,8,Andorra,AND
8,9,Angola,AGO
9,10,Antigua and Barbuda,ATG


In [15]:
show("""
    SELECT *
    FROM disease_cases
    WHERE number_rabies IS NOT NULL
    ORDER BY entity_id, year
    LIMIT 10;
""")

,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,1,1990,None,336,0,2.145772,1219845.8,87.665596,16303.112,None,None
1,1,1991,None,14,0,2.3209748,1132781.5,103.77359,17582.254,None,None
2,1,1992,None,None,0,2.6180408,1081609.6,119.9334,19786.672,None,None
3,1,1993,None,None,0,2.8531172,963817.1,127.2633,21195.934,None,37046
4,1,1994,None,None,0,2.9899871,818779.5,133.06308,21658.172,None,38735
5,1,1995,None,0,0,3.126361,725758.3,143.54527,22163.428,None,19903
6,1,1996,None,None,0,3.281437,665807.44,152.26442,22701.54,None,None
7,1,1997,None,21,0,3.446122,605853.06,160.59525,23140.863,None,4170
8,1,1998,None,65,0,3.5570247,547045.4,164.60725,23179.447,None,10000
9,1,1999,None,167,0,3.607037,502898.88,168.99359,22944.182,None,24639


**Lossless check:** joining the two normalized tables back must give exactly the original data. Both differences below must be 0.

In [16]:
show(r"""
    WITH restored AS (
        SELECT e.entity_name AS entity,
               e.entity_code AS code,
               dc.year,
               dc.number_yaws, dc.polio_cases, dc.cases_guinea_worm, dc.number_rabies,
               dc.number_malaria, dc.number_hiv, dc.number_tuberculosis,
               dc.number_smallpox, dc.number_cholera_cases
        FROM disease_cases AS dc
        JOIN entities AS e USING (entity_id)
    ),
    original AS (
        SELECT entity,
               NULLIF(code, '') AS code,
               year::SMALLINT AS year,
               NULLIF(number_yaws, '')::NUMERIC AS number_yaws,
               NULLIF(polio_cases, '')::NUMERIC AS polio_cases,
               NULLIF(cases_guinea_worm, '')::NUMERIC AS cases_guinea_worm,
               NULLIF(number_rabies, '')::NUMERIC AS number_rabies,
               NULLIF(number_malaria, '')::NUMERIC AS number_malaria,
               NULLIF(number_hiv, '')::NUMERIC AS number_hiv,
               NULLIF(number_tuberculosis, '')::NUMERIC AS number_tuberculosis,
               NULLIF(number_smallpox, '')::NUMERIC AS number_smallpox,
               NULLIF(number_cholera_cases, '')::NUMERIC AS number_cholera_cases
        FROM infectious_cases
    )
    SELECT
        (SELECT COUNT(*) FROM (SELECT * FROM original EXCEPT SELECT * FROM restored) AS t) AS missing_in_normalized,
        (SELECT COUNT(*) FROM (SELECT * FROM restored EXCEPT SELECT * FROM original) AS t) AS extra_in_normalized;
""")

,missing_in_normalized,extra_in_normalized
0,0,0


## 3. Analyze data

For every unique Entity + Code combination (identified by `entity_id`) the query calculates the average, minimum, maximum and sum of `number_rabies`, sorts by the average in descending order and returns 10 rows.

`WHERE number_rabies IS NOT NULL` is used because 19 entities have no rabies data at all, so their `AVG` is `NULL`. In PostgreSQL `ORDER BY ... DESC` puts `NULL` values first by default (`NULLS FIRST`), so without the filter the top 10 would be filled with `NULL` averages. The filter does not change the average, minimum, maximum or sum of the other entities.

In [17]:
show(r"""
    SELECT
        e.entity_id,
        e.entity_name,
        e.entity_code,
        ROUND(AVG(dc.number_rabies), 2) AS avg_rabies,
        ROUND(MIN(dc.number_rabies), 2) AS min_rabies,
        ROUND(MAX(dc.number_rabies), 2) AS max_rabies,
        ROUND(SUM(dc.number_rabies), 2) AS sum_rabies,
        COUNT(dc.number_rabies)         AS n_years
    FROM disease_cases AS dc
    JOIN entities AS e USING (entity_id)
    WHERE dc.number_rabies IS NOT NULL
    GROUP BY e.entity_id, e.entity_name, e.entity_code
    ORDER BY AVG(dc.number_rabies) DESC
    LIMIT 10;
""")

,entity_id,entity_name,entity_code,avg_rabies,min_rabies,max_rabies,sum_rabies,n_years
0,241,World,OWID_WRL,20192.37,14075.51,24744.66,605771.11,30
1,121,Lower Middle Income (WB),None,15193.96,10202.53,19182.80,455818.78,30
2,202,South Asia (WB),None,11729.89,7271.28,15361.88,351896.66,30
3,206,South-East Asia Region (WHO),None,11424.33,6806.01,15641.96,342729.81,30
4,77,G20,None,10189.05,6339.08,13164.88,305671.38,30
5,97,India,IND,8599.17,5425.87,11121.14,257975.20,30
6,209,Sub-Saharan Africa (WB),None,4748.27,4356.82,5017.98,142448.04,30
7,3,African Region (WHO),None,4649.70,4283.07,4917.78,139491.07,30
8,120,Low Income (WB),None,3568.11,3014.62,3873.53,107043.16,30
9,58,East Asia & Pacific (WB),None,3427.46,2009.94,4591.01,102823.83,30


**Note on the result:** the dataset mixes countries with aggregates (World, income groups, WHO regions, G20), so nine of the top 10 rows above are aggregates and only India is a country. Aggregates have no country code, except `World` (`OWID_WRL`). The task does not ask to exclude them, so the query above is the required result. For reference, below is the same query restricted to countries only (rows without a code and aggregates with an `OWID_` code are excluded):

In [18]:
show(r"""
    SELECT
        e.entity_id,
        e.entity_name,
        e.entity_code,
        ROUND(AVG(dc.number_rabies), 2) AS avg_rabies,
        ROUND(MIN(dc.number_rabies), 2) AS min_rabies,
        ROUND(MAX(dc.number_rabies), 2) AS max_rabies,
        ROUND(SUM(dc.number_rabies), 2) AS sum_rabies,
        COUNT(dc.number_rabies)         AS n_years
    FROM disease_cases AS dc
    JOIN entities AS e USING (entity_id)
    WHERE dc.number_rabies IS NOT NULL
      AND e.entity_code IS NOT NULL
      AND e.entity_code NOT LIKE 'OWID\_%'
    GROUP BY e.entity_id, e.entity_name, e.entity_code
    ORDER BY AVG(dc.number_rabies) DESC
    LIMIT 10;
""")

,entity_id,entity_name,entity_code,avg_rabies,min_rabies,max_rabies,sum_rabies,n_years
0,97,India,IND,8599.17,5425.87,11121.14,257975.20,30
1,163,Pakistan,PAK,1582.17,1177.14,1881.73,47465.09,30
2,151,Nigeria,NGA,1335.42,1073.95,1441.12,40062.46,30
3,42,China,CHN,1247.91,744.25,2075.59,37437.20,30
4,70,Ethiopia,ETH,1145.17,758.05,1323.40,34355.18,30
5,143,Myanmar,MMR,972.90,560.70,1449.83,29187.11,30
6,19,Bangladesh,BGD,785.71,180.40,1734.71,23571.26,30
7,146,Nepal,NPL,661.25,338.14,1041.63,19837.53,30
8,170,Philippines,PHL,618.69,358.06,898.07,18560.73,30
9,98,Indonesia,IDN,267.71,124.62,481.37,8031.44,30


## 4. Difference in years

Built-in functions: `MAKE_DATE(year, 1, 1)` creates January 1st of the year, `CURRENT_DATE` is today's date, and `EXTRACT(YEAR FROM AGE(CURRENT_DATE, ...))` is the difference between them in full years.

In [19]:
show(r"""
    SELECT
        e.entity_name,
        dc.year,
        MAKE_DATE(dc.year, 1, 1) AS year_start_date,
        CURRENT_DATE AS today,
        EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(dc.year, 1, 1)))::INTEGER AS years_diff
    FROM disease_cases AS dc
    JOIN entities AS e USING (entity_id)
    ORDER BY e.entity_name, dc.year
    LIMIT 10;
""")

,entity_name,year,year_start_date,today,years_diff
0,Afghanistan,1960,1960-01-01,2026-11-07,66
1,Afghanistan,1965,1965-01-01,2026-11-07,61
2,Afghanistan,1980,1980-01-01,2026-11-07,46
3,Afghanistan,1981,1981-01-01,2026-11-07,45
4,Afghanistan,1982,1982-01-01,2026-11-07,44
5,Afghanistan,1983,1983-01-01,2026-11-07,43
6,Afghanistan,1984,1984-01-01,2026-11-07,42
7,Afghanistan,1985,1985-01-01,2026-11-07,41
8,Afghanistan,1986,1986-01-01,2026-11-07,40
9,Afghanistan,1987,1987-01-01,2026-11-07,39


## 5. Functions

**5.1 Difference in years**

The function creates the same attribute as in the previous section: it takes a year as input and returns the difference in years between the current date and January 1st of that year (for example, 1996 → 1996-01-01 → difference between today and that date). It is marked `STABLE` because it depends on `CURRENT_DATE`, and `STRICT` so that a `NULL` year returns `NULL`.

In [20]:
run("""
    CREATE OR REPLACE FUNCTION fn_years_since(p_year INTEGER)
    RETURNS INTEGER
    LANGUAGE sql
    STABLE
    STRICT
    AS $$
        SELECT EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INTEGER;
    $$;
""")

In [21]:
show(r"""
    SELECT 
        1996 AS year_value,
        make_date(1996, 1, 1) AS year_start,
        CURRENT_DATE AS today,
        fn_years_since(1996) AS years_diff;
""")

,year_value,year_start,today,years_diff
0,1996,1996-01-01,2026-11-07,30


In [22]:
show(r"""
    SELECT 
        e.entity_name,
        dc.year,
        fn_years_since(dc.year) AS years_diff
    FROM disease_cases AS dc
    JOIN entities AS e USING (entity_id)
    ORDER BY e.entity_name, dc.year
    LIMIT 10;
""")

,entity_name,year,years_diff
0,Afghanistan,1960,66
1,Afghanistan,1965,61
2,Afghanistan,1980,46
3,Afghanistan,1981,45
4,Afghanistan,1982,44
5,Afghanistan,1983,43
6,Afghanistan,1984,42
7,Afghanistan,1985,41
8,Afghanistan,1986,40
9,Afghanistan,1987,39


**5.2 Average number of cases per period**

The function takes two arguments: the number of cases per year and an arbitrary divisor, and returns the average number of cases for the corresponding period. A zero divisor returns `NULL` instead of raising an error. The divisors used for testing are 12 (per month), 4 (per quarter) and 2 (per half-year).

The task requires using only records where the number of cases is numeric. In the normalized table the indicator columns are `NUMERIC`, and every non-numeric (empty) value was converted to `NULL` during the load (see the section 1 conclusion: `other_values = 0`). So `number_rabies IS NOT NULL` selects exactly the numeric records.

In [23]:
run(r"""
    CREATE OR REPLACE FUNCTION fn_avg_cases_per_period(p_cases_per_year NUMERIC, p_divisor NUMERIC)
    RETURNS NUMERIC
    LANGUAGE sql
    IMMUTABLE
    AS $$
        SELECT p_cases_per_year / NULLIF(p_divisor, 0);
    $$;
""")

In [24]:
show(r"""
    SELECT 
        ROUND(fn_avg_cases_per_period(1200, 12), 2) AS per_month,
        ROUND(fn_avg_cases_per_period(1200, 4), 2) AS per_quarter,
        ROUND(fn_avg_cases_per_period(1200, 2), 2) AS per_half_year,
        fn_avg_cases_per_period(NULL, 12) AS null_cases,
        fn_avg_cases_per_period(1200, 0) AS zero_divisor;
""")

,per_month,per_quarter,per_half_year,null_cases,zero_divisor
0,100.00,300.00,600.00,None,None


In [25]:
show(r"""
    SELECT e.entity_name,
           dc.year,
           fn_years_since(dc.year) AS years_diff,
           ROUND(dc.number_rabies, 4) AS cases_per_year,
           ROUND(fn_avg_cases_per_period(dc.number_rabies, 12), 4) AS avg_per_month,
           ROUND(fn_avg_cases_per_period(dc.number_rabies, 4), 4) AS avg_per_quarter,
           ROUND(fn_avg_cases_per_period(dc.number_rabies, 2), 4) AS avg_per_half_year
    FROM disease_cases AS dc
    JOIN entities AS e USING (entity_id)
    WHERE dc.number_rabies IS NOT NULL  -- numeric records only
    ORDER BY e.entity_name, dc.year
    LIMIT 10;
""")

,entity_name,year,years_diff,cases_per_year,avg_per_month,avg_per_quarter,avg_per_half_year
0,Afghanistan,1990,36,2.1458,0.1788,0.5364,1.0729
1,Afghanistan,1991,35,2.3210,0.1934,0.5802,1.1605
2,Afghanistan,1992,34,2.6180,0.2182,0.6545,1.3090
3,Afghanistan,1993,33,2.8531,0.2378,0.7133,1.4266
4,Afghanistan,1994,32,2.9900,0.2492,0.7475,1.4950
5,Afghanistan,1995,31,3.1264,0.2605,0.7816,1.5632
6,Afghanistan,1996,30,3.2814,0.2735,0.8204,1.6407
7,Afghanistan,1997,29,3.4461,0.2872,0.8615,1.7231
8,Afghanistan,1998,28,3.5570,0.2964,0.8893,1.7785
9,Afghanistan,1999,27,3.6070,0.3006,0.9018,1.8035
